# PAR

This notebook is for exploring PAR values from all relevant deployments. It was sparked by noticing that many PAR values are less than zero, and wanting to explore the PAR data values across deployments

In [2]:
from pathlib import Path

# import pandas as pd
import numpy as np
import xarray as xr

from esdglider import gcp, paths, utils

home = Path.home()
deployments = [
    "unit_1024-20250224",
    "risso-20250414",
    "stenella-20250414",
    "risso-20260128",
    "stenella-20260128",

]

# gcp.gcs_mount_bucket(
#     paths.data_out_bucket_name, 
#     f"/home/user/mnt-gcs/{paths.data_out_bucket_name}", 
#     ro=True
# )

In [3]:
# Exploration
deployment_name = "unit_1024-20250224"
glider_paths = paths.get_path_glider(
    deployment_name = deployment_name, 
    mode = "delayed", 
    home_path = home, 
)

ds_raw = xr.load_dataset(glider_paths["tsrawpath"])
display(ds_raw)


<xarray.Dataset> Size: 75MB
Dimensions:                       (time: 138146)
Coordinates:
  * time                          (time) datetime64[ns] 1MB 2025-02-24T22:18:...
    latitude                      (time) float64 1MB nan nan nan ... nan nan nan
    longitude                     (time) float64 1MB nan nan nan ... nan nan nan
Data variables: (12/60)
    m_depth                       (time) float64 1MB nan nan nan ... nan nan nan
    m_heading                     (time) float64 1MB nan nan nan ... nan nan nan
    m_pitch                       (time) float64 1MB nan nan nan ... nan nan nan
    m_roll                        (time) float64 1MB nan nan nan ... nan nan nan
    m_tot_num_inflections         (time) float64 1MB nan nan nan ... nan nan nan
    m_altitude                    (time) float64 1MB nan nan nan ... nan nan nan
    ...                            ...
    sci_bsipar_supply_volts       (time) float64 1MB nan nan nan ... nan nan nan
    distance_over_ground          (time) float64 1MB nan nan nan ... nan nan nan
    source_filename               (time) <U12 7MB '02990000.dbd' ... '0299002...
    depth_ctd                     (time) float64 1MB nan nan nan ... nan nan nan
    profile_index                 (time) float64 1MB 0.5 0.5 0.5 ... 54.5 54.5
    profile_direction             (time) float64 1MB nan nan nan ... nan nan nan
Attributes: (12/62)
    Conventions:               CF-1.8
    Metadata_Conventions:      Unidata Dataset Discovery v1.0, COARDS, CF-1.8
    acknowledgment:            This work was supported by funding from NOAA.
    cdm_data_type:             Trajectory
    comment:                   This glider leaked from the aft section approx...
    contributor_name:          Christian Reiss, George Watters, Anthony Cossi...
    ...                        ...
    summary:                   These data are part of the NOAA Ecosystem Scie...
    time_coverage_end:         20250226T055337
    time_coverage_start:       20250224T221800
    title:                     unit_1024-20250224T2218
    transmission_system:       IRIDIUM
    wmo_id:

In [ ]:
for deployment_name in deployments:
    print(f"Processing deployment: {deployment_name} --------------------")
    glider_paths = paths.get_path_glider(
        deployment_name = deployment_name, 
        mode = "delayed", 
        home_path = home, 
    )

    par_var = "sci_bsipar_par"
    par_volts_var = "sci_bsipar_sensor_volts"

    try: 
        with xr.open_dataset(glider_paths["tsrawpath"]) as ds:
            sn, _ = utils.get_instrument_sn_date(ds, "instrument_par")
            print(f"PAR instrument serial number: {sn}")

            par_orig = ds.to_pandas()
            par_orig = par_orig[["sci_water_pressure", par_var, par_volts_var]]
            print(f"Number of total raw timeseries timestamps: {len(par_orig)}")

            par = par_orig[par_orig[par_var].notna()]
            print(f"Number of non-nan par values: {len(par)}")
            par.to_csv(f"/home/user/par/{deployment_name}_par.csv")

            print(f"Number of par values less than 0: {len(par[par[par_var] < 0])}")
            print(f"Number of par sensor voltage values less than 0: {len(par[par[par_volts_var] < 0])}")
            print(f"par min/max values: {par[par_var].min()} / {par[par_var].max()}")
            print(f"Number of par values greater than 3000 / 3500: {len(par[par[par_var] > 3000])} / {len(par[par[par_var] > 3500])}")

    except FileNotFoundError:
        print(f"File not found for deployment: {deployment_name}")

In [ ]:
glider_paths_curr = paths.get_path_glider(
    deployment_name = deployments[1], 
    mode = "delayed", 
    home_path = home, 
)

ds = xr.load_dataset(glider_paths_curr["tsrawpath"])
# ds

df = ds.to_pandas()

df["m_depth_interp"] = df["m_depth"].interpolate(method="time", limit_area="inside")
df["m_depth_interp"].describe()
df = df[["m_depth", "m_depth_interp", "sci_bsipar_sensor_volts", "sci_bsipar_par"]]

df


In [ ]:
df_filt = df[(df['m_depth_interp'] > 500) & df['sci_bsipar_sensor_volts'].notna()]
df_filt = df_filt[["m_depth", "m_depth_interp", "sci_bsipar_sensor_volts", "sci_bsipar_par"]]
print(f"Number of points: {len(df_filt)}")
df_filt

In [ ]:
par_volt_mean = df_filt["sci_bsipar_sensor_volts"].mean()
print(par_volt_mean)
print(len(df))
print(np.count_nonzero(df["sci_bsipar_sensor_volts"] < par_volt_mean))